In [ ]:
import os 
from glob import glob
import pandas as pd

In [ ]:
# 특정 경로에서 파일의 목록을 가져오는 기능 
os.listdir("./review")

In [ ]:
# glob 라이브러리 사용
# 장점 : 파일의 경로와 파일 명이 동시에 출력 
#       특정 확장자만 목록을 불러올수 있다. 
json_list = glob("./review/*.json")

In [ ]:
# json_list를 이용하여 여러 파일들을 하나의 데이터프레임으로 결합 

# 비어있는 데이터프레임을 생성 
total_df = pd.DataFrame()

for file_path in json_list:
    # print(file_path)
    # break
    df = pd.read_json(file_path)
    # df를 total_df에 단순 행 결합 
    total_df = pd.concat( [total_df, df], axis = 0 )
total_df.reset_index(drop = True, inplace=True)
total_df.info()

In [ ]:
total_df.head(2)

In [ ]:
aspect_df = pd.DataFrame(sum(total_df['Aspects'], []))

In [ ]:
aspect_df.head()

In [ ]:
# 결측치가 존재하는가?
aspect_df.info()

In [ ]:
aspect_df = aspect_df.map(lambda x : x.strip())

In [ ]:
(aspect_df == '').sum()

In [ ]:
# 종속 변수들의 데이터의 빈도수를 확인 
aspect_df['Aspect'].value_counts()

In [ ]:
aspect_df['SentimentPolarity'].value_counts()

In [ ]:
# 독립 변수에서 중복된 데이터가 존재하면 제거 
aspect_df.drop_duplicates('SentimentText', inplace=True)

In [ ]:
aspect_df['SentimentPolarity'].value_counts()

In [ ]:
# 인덱스 초기화
aspect_df.reset_index(drop=True, inplace=True)

In [ ]:
# 독립 변수를 토큰화 -> 벡터화 
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# 토큰화 과정에서 특정 품사들만 사용
# 명사, 동사, 형용사, 부사

okt = Okt()
allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb']

def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if (pos in allow_pos) & (len(word) >= 2):
            result.append(word)
    return result

tokenize(
    aspect_df.loc[0, 'SentimentText']
)

In [112]:
vec = TfidfVectorizer(
    tokenizer= tokenize, 
    ngram_range= (1, 2), 
    min_df= 3, 
    max_df = 0.8, 
    max_features= 3000
)

In [ ]:
aspect_df.columns

In [ ]:
X = aspect_df['SentimentText'].values
y1 = aspect_df['Aspect'].values
y2 = aspect_df['SentimentPolarity'].values

In [ ]:
X_vec = vec.fit_transform(X)

In [ ]:
X_vec.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le1 = LabelEncoder()
y1_le = le1.fit_transform(y1) 
le2 = LabelEncoder()
y2_le = le2.fit_transform(y2)

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
svc1 = LinearSVC(class_weight='balanced', random_state=42)
svc2 = LinearSVC(class_weight='balanced', random_state=42)

In [62]:
svc1.fit(X_vec, y1_le)
svc2.fit(X_vec, y2_le)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [ ]:
X_vec[-500:]

In [ ]:
y1_le[-500:]

In [ ]:
pred_1 = svc1.predict(X_vec[-500:])
pred_2 = svc2.predict(X_vec[-500:])

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(pred_1, y1_le[-500:]))

In [ ]:
print(classification_report(pred_2, y2_le[-500:]))

- Ascept 컬럼과 SentimentPolarity 컬럼의 데이터를 결합 
- LabelEncoder를 이용하여 수치로 변환 
- Kfold, Pipe, GridsearchCV를 활용
- 계층화 폴드화는 5개 
- 벡터화, 모델 학습을 파이프라인으로 생성 
- 파라미터 조합은 벡터화에서 max_features의 조합은 (3000, None)
- SVC 모델의 조합은 C 값을 (1.0, 2.0)
- 최적의 모델의 스코어를 확인 

In [ ]:
aspect_df.info()

In [ ]:
aspect_df['Aspect'] + '_' +aspect_df['SentimentPolarity']

In [42]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline

In [43]:
X = aspect_df['SentimentText'].values
y = (aspect_df['Aspect'] + '_' + aspect_df['SentimentPolarity']).values

In [44]:
le = LabelEncoder()
y_le = le.fit_transform(y)

In [45]:
y_le

array([17, 12,  5, ..., 17, 17, 19], shape=(10467,))

In [46]:
pipe = Pipeline(
    [
        ('vector', vec), 
        ('model', svc1)
    ]
)
cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)
params = {
    'vector__max_features' : [3000, None], 
    'model__C' : [1.0, 2.0]
}

In [47]:
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    cv = cv
)

In [48]:
grid.fit(X, y_le)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [1.0, 2.0], 'vector__max_features': [3000, None]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 :

In [49]:
grid.best_score_

np.float64(0.7412812800753501)

In [50]:
# 종속 변수를 2차원의 데이터로 그대로 활용 
X = aspect_df['SentimentText'].values
le = LabelEncoder()
aspect_df['Aspect_le'] = le.fit_transform(aspect_df['Aspect'])
aspect_df['SentimentPolarity'] = aspect_df['SentimentPolarity'].astype(int)
y = aspect_df[ ['Aspect_le', 'SentimentPolarity'] ].values

In [52]:
y.shape

(10467, 2)

In [53]:
X.shape

(10467,)

In [117]:
vec = TfidfVectorizer(
    tokenizer=tokenize, 
    ngram_range= (1,2), 
    min_df = 3, 
    max_df = 0.8, 
    lowercase= False
)

In [118]:
X_vec = vec.fit_transform(X)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
svc = LinearSVC(random_state=42, class_weight='balanced')

In [59]:
from sklearn.multioutput import MultiOutputClassifier

In [60]:
# MultiOutputClassifier는 분류모델에서 종속변수가 2차원 이상인 경우 사용하는 객체 
multi_model = MultiOutputClassifier(svc)

In [74]:
multi_model.fit(X_vec, y)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LinearSVC(random_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_

In [73]:
print(type(y[0][0]), type(y[0][1]))

<class 'numpy.int64'> <class 'numpy.int64'>


In [89]:
from sklearn.model_selection import KFold
cv2 = KFold(n_splits=5, random_state=42, shuffle=True)

In [92]:
# 쉬는시간에 확인 

pipe = Pipeline(
    [
        ('vector', vec),
        ('model', multi_model)
    ]
)
params = {
    'model__estimator__C' : [0.8, 1.0]
}
grid = GridSearchCV(
    estimator= pipe, 
    cv = cv2, 
    param_grid= params, 
    scoring= 'accuracy'
)

In [ ]:
grid.fit(X, y)

In [80]:
pred = multi_model.predict(X_vec[-20:])

In [ ]:
pred

In [83]:
pred_df = pd.DataFrame(
    pred, columns = ['aspect', 'polarity']
)
pred_df['text'] = X[-20:]

In [85]:
pred_df['aspect'] = le.inverse_transform(pred_df['aspect'])

In [86]:
pred_df

,aspect,polarity,text
0,디자인,1,디자인이 너무너무 예쁩니다.
1,색상,1,색상도 고급스러워서 마음에 들구요.
2,마감,1,바느질도 꼼꼼하게 잘 되어 있어요.
3,촉감,1,털 감촉도 아주 부드러워요~
4,핏,1,입어 보면 핏이 더 예쁜 자켓이예요~
5,가격,1,가격 대비 가성비 좋은 제품이예요~
6,색상,1,색상도 좋구요~
7,사이즈,1,모자 크기도 적당해서 마음에 들어요.
8,디자인,1,모자가 있다 보니 밍크 특유의 올드함이 없네요.
9,디자인,1,디자인 마음에 들어요.


In [94]:
# 긴 문단을 문장 별로 나눠주기 위해 Kkma로드 
from konlpy.tag import Kkma

In [95]:
kkma = Kkma()

In [97]:
file_list = glob("./test/*.json")
file_list

['./test\\1-1.여성의류(1).json',
 './test\\1-1.여성의류(2).json',
 './test\\1-1.여성의류(3).json']

In [98]:
test_df = pd.DataFrame()

for file in file_list:
    df = pd.read_json(file)
    test_df = pd.concat( [test_df, df], axis=0 )

test_df.reset_index(drop=True, inplace=True)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            300 non-null    int64  
 1   RawText          300 non-null    object 
 2   Source           300 non-null    object 
 3   Domain           300 non-null    object 
 4   MainCategory     300 non-null    object 
 5   ProductName      300 non-null    object 
 6   ReviewScore      300 non-null    int64  
 7   Syllable         300 non-null    int64  
 8   Word             300 non-null    int64  
 9   RDate            300 non-null    int64  
 10  GeneralPolarity  298 non-null    float64
 11  Aspects          300 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 28.3+ KB


In [108]:
sample_text = test_df.loc[12, 'RawText']
sample_text

'털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고  하루입고벗었더니 겨드랑이같은곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도없이 손에 털뭉치???? 들이 묻어나서 포기했습니다 이게불량인가요 왜이렇게 만드신건가요 옷디자인은 너무맘에드는데 털빠짐때매 옷이아니네여'

In [109]:
# sample_text를 문장별로 나눠준다. 
text_list = kkma.sentences(sample_text)
text_list

['털이 엄청나게 빠집니다',
 '면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도 없이 손에 털 뭉치???? 들이 묻어나서 포기했습니다',
 '이게 불량인가요',
 '왜 이렇게 만드신 건가요',
 '옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여']

In [ ]:
# text_list를 벡터화  -> 모델을 이용하여 예측 



In [121]:
pred = grid.predict(text_list)

In [122]:
pred

array([[ 1,  1],
       [15,  1],
       [15, -1],
       [ 2,  1],
       [ 5,  1]])

In [123]:
pred_df = pd.DataFrame(pred, columns = ['aspect', 'polarity'])
pred_df['text'] = text_list

pred_df

,aspect,polarity,text
0,1,1,털이 엄청나게 빠집니다
1,15,1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...
2,15,-1,이게 불량인가요
3,2,1,왜 이렇게 만드신 건가요
4,5,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여


In [124]:
pred_df['aspect'] = le.inverse_transform(pred_df['aspect'])
pred_df

,aspect,polarity,text
0,기능,1,털이 엄청나게 빠집니다
1,품질,1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...
2,품질,-1,이게 불량인가요
3,길이,1,왜 이렇게 만드신 건가요
4,디자인,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여


In [125]:
pred_df['RawText'] = sample_text
pred_df

,aspect,polarity,text,RawText
0,기능,1,털이 엄청나게 빠집니다,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
1,품질,1,면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니...,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
2,품질,-1,이게 불량인가요,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
3,길이,1,왜 이렇게 만드신 건가요,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...
4,디자인,1,옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여,털이 엄청나게 빠집니다 면 바바리에 엄청나게 묻어나고 하루입고벗었더니 겨드랑이같은...


In [127]:
# test_df에서 RawText의 데이터들을 문장 별로 나눠준다. 
# raw_list 리뷰 문단을 문장으로 나눈 리스트를 담기 위한 공간 
raw_list = []
# raw_dict 리뷰 문단마다 index를 키 값으로 value는 리뷰 문단 
raw_dict = {}
for i in range(len(test_df)):
    raw_list.append(
        kkma.sentences(
            test_df.loc[i, 'RawText']
        )
    )
    raw_dict[i] = test_df.loc[i, 'RawText']

In [ ]:
# raw_list
sentence_df = pd.DataFrame()
for idx, raw in enumerate(raw_list):
    pred = grid.predict(raw)
    temp_df = pd.DataFrame(pred, columns = ['Pred_Aspect', 'Pred_Polarity'])

[['가격이 착하고 디자인이 예쁩니다'],
 ['싸고 디자인이 예뻐요. .', '정말 가 성비 가심 비 입니다'],
 ['편하고 디자인이 예뻐요', '가격도 좋아요', '시 원해요 빨리 마르고 이것만 입게 되요'],
 ['너무 착한 가격에 감사합니다', '윈하는 색은 없지만'],
 ['가격이 너무 좋아서 블랙 구매했습니다', '그런데 소재도 맘에 들어 흰색도 구매했습니다'],
 ['외출할 때 입기에는 재질도 디자인도 좀~ 그러네요', '그냥 집에서 편하게 입을 수는 있을 것 같아요'],
 ['특히 자켓이 넘 작고 짧으네요. 상의는 좀 작구 바지는 잘 맞아요.'],
 ['싸고 품질이 좋아요', 'ᆢ 사이즈는 좀 큰 듯하구요', 'ᆢ 반 사이즈는 내려서 주문 하심 좋을 듯해요'],
 ['바지가 너무 편하고 좋아요', '티셔츠는 여름에 잘 입을 듯 베스트는 디자인은 별로 지만 가격이 착해서 만족'],
 ['디자인이 예뻐요.', '사이즈 잘 맞습니다.'],
 ['사이즈 딱 맞구요.', '너무 예쁘고 얇지도 두껍지도 않으네요.'],
 ['저렴하게 구입해서 겨울에 잘 입을 거 같아요'],
 ['털이 엄청나게 빠집니다',
  '면 바바리에 엄청나게 묻어나고 하루 입고 벗었더니 겨드랑이 같은 곳도 엄청 뭉처있니에요 그래서 세탁을 했더니 끝도 없이 손에 털 뭉치???? 들이 묻어나서 포기했습니다',
  '이게 불량인가요',
  '왜 이렇게 만드신 건가요',
  '옷 디자인은 너무 맘에 드는데 털 빠짐 때 매 옷이 아니 네 여'],
 ['디자인이 여성스러운 분위기가 납 니다 구겨져 와서 다림질 필요했어요'],
 ['99 사이즈 입는데 88 사이즈 잘 맞고~ 기장이 살짝 짧은 듯 하지만 밝은 색도 있고~ 까실까실 하니 좋으네요~'],
 ['4 종 마음에 들어요', '색깔도 이쁘고 여름 니트 입니다'],
 ['색상이 예쁘고 구김이 없어요'],
 ['색상 디자인 예쁘고 맘에 들어요'],
 ['가 성비 갑입니다',
  '가로 패턴이라 더 커 보이면 어쩌나 싶었는데 칼라고 디자인이고 다